# Transformer Architecture in Practice


Lecture 10 built the transformer from UDL chapter 12: dot-product
self-attention, multi-head attention, the transformer layer, and the three model
types. This lecture is about what sits around and inside that block in a model
you can download today. The first two sections carry over from the previous
version of this lecture: how text becomes token IDs (tokenization) and how a
probability vector becomes the next token (decoding). The rest is new: the
anatomy of a 2026 decoder-only model (pre-norm, RMSNorm, RoPE, SwiGLU, grouped-query
attention, sliding windows, latent attention, mixture of experts), and a
walk through the `config.json` of a real open-weight model, from which we count
its parameters and size its KV cache. The quizzable calculations are the KV-cache
bytes per token, the active-parameter count of a mixture of experts, and the
parameter count of a model from its config. Figures are from Prince,
*[Understanding Deep Learning](https://udlbook.github.io/udlbook/)*, unless
another source is cited. UDL does not cover the newer components; for those the
notation follows the cited papers, adapted to UDL's column-vector convention
($\mathbf{h}$ hidden vector, $\boldsymbol{\Omega}$ weights, $\boldsymbol{\beta}$ biases).
The tokenization and decoding code cells are adapted from the UDL notebooks (MIT license,
[github.com/udlbook/udlbook/tree/main/Notebooks](https://github.com/udlbook/udlbook/tree/main/Notebooks)).


## Today's roadmap

1. Tokenization: from text to token IDs, and why BPE
2. Decoding: from a probability vector to the next token
3. The modern decoder-only block: pre-norm, RMSNorm, RoPE, SwiGLU, no biases
4. Attention variants: the KV cache, MQA/GQA, sliding windows, latent attention
5. Mixture of experts: routing, load balancing, active vs. total parameters
6. Reading a real config: Qwen3-0.6B, field by field, with a parameter count
7. Context length: why it costs what it costs
8. Summary


# Tokenization

A transformer operates on vectors, not on raw text. Before the embedding layer
of lecture 10 can do anything, the text has to be cut into a sequence of
integers from a fixed vocabulary. This section covers how that vocabulary is
built and why it is built that way.


## From text to vectors

![](figs/11-Transformer-Architecture-in-Practice/nlp-preprocessing-pipeline.png)

A **token** is a small chunk of text, one or more bytes, drawn from a fixed
vocabulary $\mathcal{V}$ of a few tens of thousands to a few hundred thousand entries.

- The **tokenizer** maps a string to a sequence of token IDs (encode) and back (decode).
- Input text is split greedily: at each point, the longest vocabulary entry that matches.
- The vocabulary itself is also built greedily, by repeatedly merging the most frequent
  adjacent pair (byte-pair encoding, below).
- The model never sees characters. Everything it knows about spelling, arithmetic
  and non-English scripts is filtered through this step.


## Why tokens, and not bits, bytes, characters or words?

The output layer is a softmax over $\mathcal{V}$, so every candidate unit costs
one output row, and the sequence length is the number of units.

- **Bits or bytes**: tiny vocabulary, but sequences get 4 to 10 times longer than with
  tokens, and a single byte of UTF-8 carries little meaning on its own.
- **Characters**: fine for English; Unicode has about 160,000 of them.
- **Words**: there are more words than characters, new ones appear every day, and
  "running", "runs" and "ran" would share nothing.
- **Subword tokens**: a vocabulary of fixed size that covers *any* string
  (every byte is a token, so there is no out-of-vocabulary), with common words and
  word pieces as single tokens.


## Unicode and UTF-8

- **Unicode** assigns every character a *code point* U+0000 to U+10FFFF:
  $17 \times 2^{16} = 1{,}114{,}112$ possible entries, of which about 160,000 are assigned
  (Unicode 17.0, 2025), covering over 170 scripts plus symbols and emoji.
- **UTF-8** (Unicode Transformation Format, 8-bit) is a *variable-length encoding* of
  code points into one to four bytes. The first 128 code points are ASCII in one byte.
- Byte-level BPE runs on the UTF-8 bytes, so its base vocabulary is the 256 byte values.


| First code point | Last code point | Byte 1 | Byte 2 | Byte 3 | Byte 4 | Coverage |
|---:|---:|---|---|---|---|---|
| U+0000 | U+007F | `0xxxxxxx` | | | | 128 code points, 1 byte. ASCII |
| U+0080 | U+07FF | `110xxxxx` | `10xxxxxx` | | | 1,920 code points, 2 bytes. Remainder of almost all Latin-script alphabets, plus IPA extensions, Greek, Cyrillic, Coptic, Armenian, Hebrew, Arabic, Syriac, Thaana and N'Ko, and combining diacritical marks |
| U+0800 | U+FFFF | `1110xxxx` | `10xxxxxx` | `10xxxxxx` | | 61,440 code points, 3 bytes. Rest of the Basic Multilingual Plane, including Chinese, Japanese and Korean characters |
| U+010000 | U+10FFFF | `11110xxx` | `10xxxxxx` | `10xxxxxx` | `10xxxxxx` | 1,048,576 code points, 4 bytes. Emoji, historic scripts, math symbols |

: Code point to UTF-8 conversion


Sources: [Wikipedia, Unicode](https://en.wikipedia.org/wiki/Unicode);
[Wikipedia, UTF-8](https://en.wikipedia.org/wiki/UTF-8).

## Byte-pair encoding (BPE)

```text
vocabulary <- the 256 byte values
while |vocabulary| < target size:
    count every adjacent pair of tokens in the corpus
    merge the most frequent pair into one new token
    add it to the vocabulary; record the merge rule
return vocabulary and the ordered list of merges
```

Encoding a new string replays the merge rules in order; decoding concatenates
the bytes. Every string is representable because every byte is a token.


- Introduced for NMT by [Sennrich et al. 2016](https://arxiv.org/abs/1508.07909);
  made *byte-level* by GPT-2 ([Radford et al. 2019](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)).
- A regex **pre-split** stops merges from crossing word, number and punctuation
  boundaries, e.g. GPT-4's pattern keeps digits in groups of at most three and
  attaches a leading space to words.
- [SentencePiece](https://arxiv.org/abs/1808.06226) (Gemma, early Llama) runs the
  same idea on Unicode code points instead of bytes.
- Vocabulary sizes in 2026: Llama 3 128K, Qwen3 about 152K, Gemma 3 262K.



## BPE worked example


![Figure 12.8 of Prince, *Understanding Deep Learning*.](figs/11-Transformer-Architecture-in-Practice/bpe-example.png)

- a\) Start from a minimal vocabulary: a subset of the lower-case Latin alphabet and the space `_`.
- b\) Find the most frequent adjacent pair, `se` here, and make it a token.
- c\) Next most frequent pair: `e_`.
- d\), e\) Continue until the vocabulary reaches its size limit.
- f\) The number of distinct tokens first rises, then falls as merges absorb the single characters.

The same procedure in code, on the same text. Each word is a tuple of tokens
ending in `_`, stored with its count, so a merge only ever joins tokens inside
one word (the role of the pre-split). Ties go to the pair seen first. Try
`num_merges = 10`, or your own text.

In [ ]:
# Adapted from UDL notebook 12.3 (MIT)
from collections import Counter

text = ("a sailor went to sea sea sea "
        "to see what he could see see see "
        "but all that he could see see see "
        "was the bottom of the deep blue sea sea sea")
num_merges = 50

words = Counter(tuple(w) + ("_",) for w in text.split())   # ('s','e','a','_'): 6, ...

def token_counts(words):
    counts = Counter()
    for word, freq in words.items():
        for tok in word:
            counts[tok] += freq
    return counts

def pair_counts(words):
    pairs = Counter()
    for word, freq in words.items():
        for a, b in zip(word, word[1:]):
            pairs[a, b] += freq
    return pairs

def merge(pair, words):
    out = Counter()
    for word, freq in words.items():
        new, i = [], 0
        while i < len(word):
            if word[i:i + 2] == pair:
                new.append(word[i] + word[i + 1])
                i += 2
            else:
                new.append(word[i])
                i += 1
        out[tuple(new)] += freq
    return out

print("start:", dict(token_counts(words).most_common(5)), "...")
n_tokens = [len(token_counts(words))]
for step in range(1, num_merges + 1):
    pairs = pair_counts(words)
    if not pairs:                         # every word is a single token
        break
    best = max(pairs, key=pairs.get)
    words = merge(best, words)
    n_tokens.append(len(token_counts(words)))
    if step <= 5:
        print(f"merge {step}: {best[0]!r} + {best[1]!r} (count {pairs[best]})")

print("final tokens:", dict(token_counts(words).most_common()))
print("distinct tokens after each merge:", n_tokens)

## Tokenizers in practice: the same Python, two vocabularies

![GPT-2 tokenizer (`gpt2`), 146 tokens. Source: <https://tiktokenizer.vercel.app/>](figs/11-Transformer-Architecture-in-Practice/tiktokenizer-gpt2-python.png)


![GPT-4 tokenizer (`cl100k_base`), 96 tokens. Source: <https://tiktokenizer.vercel.app/>](figs/11-Transformer-Architecture-in-Practice/tiktokenizer-gpt4-python.png)


GPT-2's vocabulary makes every indentation space its own token; GPT-4's merges
runs of spaces. Same model architecture, one third fewer tokens per line of code.

## Tokenization matters

From the GPT-4o announcement: a larger vocabulary (200K) gives fewer tokens for
non-English text, about 1.1$\times$ for English, French and Spanish up to 2 to
4.4$\times$ for Arabic, Hindi, Telugu and Gujarati, so the same context window
holds more text and each request costs less ([OpenAI 2024](https://openai.com/index/hello-gpt-4o/));
the figure shows five of the rows.

Karpathy's list of "model problems" that are really tokenizer problems
([minbpe lecture](https://github.com/karpathy/minbpe/blob/master/lecture.md)):

- Why can't the LLM spell, or reverse a string? Tokenization.
- Why is it worse at Japanese? Why is it bad at arithmetic? Tokenization.
- Why did GPT-2 struggle with Python? Tokenization.
- Why does it break on the string `<|endoftext|>` or on "SolidGoldMagikarp"? Tokenization.


![Source: [OpenAI, Hello GPT-4o](https://openai.com/index/hello-gpt-4o/).](figs/11-Transformer-Architecture-in-Practice/gpt4o-tokenizer-language-gains.png)


## Learned embeddings

![Figure 12.9 of Prince, *Understanding Deep Learning*.](figs/11-Transformer-Architecture-in-Practice/vocabulary-embeddings.png)


Each token ID selects one column of a learned $D \times |\mathcal{V}|$ matrix
$\boldsymbol{\Omega}_e$: $\mathbf{X} = \boldsymbol{\Omega}_e \mathbf{T}$ with
$\mathbf{T}$ one-hot.

```python
nn.Embedding(vocab_size, hidden_size)
```

- Qwen3-0.6B: $|\mathcal{V}| = 151{,}936$, $D = 1024$, so **156M** of its 596M
  parameters are this one matrix (26%).
- The output layer is the transpose shape, $|\mathcal{V}| \times D$; small models
  **tie** the two (one matrix, used twice). More on this later.



# Decoding strategies

At the other end of the model, a decoder outputs a probability for every token in
the vocabulary at every position. This section compares the ways of turning that
distribution into the next token: deterministic, sampled, and searched.


## From a probability vector to a token

![Figure 12.12 of Prince, *Understanding Deep Learning*.](figs/11-Transformer-Architecture-in-Practice/decoder-encoder-decoder-outputs.png)

At each step the model gives logits $\mathbf{z} \in \mathbb{R}^{|\mathcal{V}|}$ and
$\mathrm{Pr}(y_t = w \mid \mathbf{y}_{<t}) = \mathrm{softmax}[\mathbf{z}]_w$. How
should we pick $y_t$? The trade-off is **accuracy versus diversity**, and the
answer differs for translation, code and chat.

## Greedy decoding

Take the most likely token at every step:

$$
\hat{y}_t = \underset{w \in \mathcal{V}}{\operatorname{argmax}} \;
\mathrm{Pr}(y_t = w \mid \hat{\mathbf{y}}_{<t}, \mathbf{x}, \boldsymbol{\phi})
$$

```python
next_id = logits.argmax(dim=-1)
```

- Deterministic: same prompt, same output. Cheap.
- Locally optimal, not globally: a high-probability first token may lead
  only to low-probability continuations.
- Tends to loop ("I think that I think that I think that ...") and reads as generic.


## Sampling, and temperature

Sample $y_t$ from the distribution instead of taking its mode. A **temperature**
$T$ rescales the logits first:

$$
\mathrm{Pr}(y_t = w) = \frac{\exp[z_w / T]}{\sum_{w' \in \mathcal{V}} \exp[z_{w'} / T]}
$$

- $T \to 0$: greedy. $T = 1$: the model's own distribution. $T > 1$: flatter, more random.
- Sampling adds diversity, but will occasionally pick from the long tail of the
  distribution, producing odd word choices.
- Temperature changes the *shape* but not the *support*: the tail is still there.
  The next two methods cut it off.


![](figs/11-Transformer-Architecture-in-Practice/sampling-sketch.png)


## Top-$k$ and nucleus (top-$p$) sampling

**Top-$k$** ([Fan et al. 2018](https://arxiv.org/abs/1805.04833))

1. Sort tokens by probability.
2. Keep the $k$ most probable; discard the rest.
3. Renormalize to sum to 1; sample.

$k = 1$ is greedy. A fixed $k$ ignores the shape of the distribution: when the
model is confident, $k = 50$ still admits 49 bad options; when it is uncertain,
$k = 50$ may cut off good ones.


**Nucleus** ([Holtzman et al. 2020](https://arxiv.org/abs/1904.09751))

Keep the smallest set $\mathcal{V}^{(p)}$ whose probability mass reaches $p$:

$$
\sum_{w \in \mathcal{V}^{(p)}} \mathrm{Pr}(w \mid \mathbf{y}_{<t}) \geq p
$$

then renormalize and sample. The set adapts to the distribution: a few tokens
when the model is sure, many when it is not.


In practice these stack: temperature, then top-$k$, then top-$p$. The Qwen3
model card recommends $T = 0.6$, top-$p = 0.95$, top-$k = 20$ for thinking mode
and warns against greedy decoding, which causes repetition
([Qwen3-0.6B model card](https://huggingface.co/Qwen/Qwen3-0.6B)).


## Beam search: the idea

Greedy maximizes each step; we want the most probable *sequence*.

$\mathcal{V} = \{\text{yes}, \text{ok}, \langle\text{eos}\rangle\}$; maximize
$\mathrm{Pr}(t_1, t_2, t_3)$.

Greedy: $0.5 \times 0.4 \times 1.0 = 0.20$

Optimal: $0.4 \times 0.7 \times 1.0 = 0.28$

Exhaustive search is $|\mathcal{V}|^N$. **Beam search** keeps the best $k$
partial sequences (the *beam width*) at every step and extends only those.


![Source: [Jurafsky and Martin, *Speech and Language Processing*](https://web.stanford.edu/~jurafsky/slp3/).](figs/11-Transformer-Architecture-in-Practice/beam-search-tree.png) for three steps t1, t2, t3. From start: ok with probability .4, yes .5, EOS .1. From ok: ok .7, yes .2, EOS .1. From yes: ok .3, yes .4, EOS .3. Every second-step ok or yes is followed by EOS with probability 1.0. The red labels give p(t1|start), p(t2|t1) and p(t3|t1,t2)." width="100%"}


## Beam search with $k = 2$


![Source: [Jurafsky and Martin, *Speech and Language Processing*](https://web.stanford.edu/~jurafsky/slp3/).](figs/11-Transformer-Architecture-in-Practice/beam-search.png)

1. Work in *log probabilities* and add. Keep the top 2 tokens.
2. Extend each of the 2 by its 2 best continuations, accumulating log probs:
   $-1.6 - .69 = -2.3$, $-1.6 - 2.3 = -3.9$, $-.92 - .69 = -1.6$,
   $-.92 - 1.2 = -2.1$. Keep the best 2 of the 4.
3. Extend again from each survivor; keep the 2 highest cumulative log probs.
4. One path reaches EOS, but a lower-scoring path is still open.
5. Two completed paths tie at $-2.7$; pick one. Longer paths accumulate more
   negative terms, so implementations normalize by length.

The decoders of this section on toy distributions. The first part applies
temperature, top-$k$ and top-$p$ to a fixed 10-token logit vector and samples
from each; the second runs beam search on the yes/ok tree of the previous slide,
where a beam of width 1 is greedy decoding. Change `T`, `k` and `p`, or the
beam widths.

In [ ]:
# Adapted from UDL notebook 12.4 (MIT)
import numpy as np

rng = np.random.default_rng(0)
vocab = np.array(["the", "a", "my", "this", "one", "his", "an", "that", "some", "zebra"])
logits = np.array([3.0, 2.5, 1.8, 1.2, 0.9, 0.5, 0.2, 0.0, -0.5, -2.0])
T, k, p = 0.7, 3, 0.9

def softmax(z):
    e = np.exp(z - z.max())
    return e / e.sum()

def top_k(probs, k):
    keep = np.zeros_like(probs)
    idx = np.argsort(probs)[::-1][:k]
    keep[idx] = probs[idx]
    return keep / keep.sum()

def top_p(probs, p):
    order = np.argsort(probs)[::-1]
    n_keep = np.searchsorted(np.cumsum(probs[order]), p) + 1   # smallest set with mass >= p
    keep = np.zeros_like(probs)
    keep[order[:n_keep]] = probs[order[:n_keep]]
    return keep / keep.sum()

dists = {"T = 1": softmax(logits), f"T = {T}": softmax(logits / T),
         f"top-k, k = {k}": top_k(softmax(logits), k), f"top-p, p = {p}": top_p(softmax(logits), p)}
print(f"{'':>14}" + "".join(f"{w:>7}" for w in vocab))
for name, probs in dists.items():
    print(f"{name:>14}" + "".join(f"{q:7.3f}" for q in probs))
print(f"{'greedy':>14} always: {vocab[logits.argmax()]}")
for name, probs in dists.items():
    print(f"{name:>14} samples: {' '.join(rng.choice(vocab, size=8, p=probs))}")

def next_probs(prefix):                   # Pr(next token | prefix), the tree on the slide
    if len(prefix) == 2:
        return {"<eos>": 1.0}
    if not prefix:
        return {"ok": 0.4, "yes": 0.5, "<eos>": 0.1}
    return {"ok": {"ok": 0.7, "yes": 0.2, "<eos>": 0.1},
            "yes": {"ok": 0.3, "yes": 0.4, "<eos>": 0.3}}[prefix[-1]]

def beam_search(width):
    beams, done = [((), 1.0)], []
    while beams:
        candidates = sorted(((seq + (w,), prob * q) for seq, prob in beams
                             for w, q in next_probs(seq).items()), key=lambda c: -c[1])
        beams = []
        for seq, prob in candidates[:width]:      # finished paths are set aside
            (done if seq[-1] == "<eos>" else beams).append((seq, prob))
    return max(done, key=lambda c: c[1])

print()
for width in [1, 2]:
    seq, prob = beam_search(width)
    print(f"beam width {width}: {' '.join(seq):<14} Pr = {prob:.2f}")

## Which decoder, when

| Method | Deterministic? | Cost per step | Where it is used |
|---|---|---|---|
| Greedy | yes | 1 forward | evaluation harnesses; short structured outputs |
| Beam search ($k$) | yes | $k$ forwards, $k$ KV caches | machine translation, speech recognition |
| Temperature sampling | no | 1 forward | chat, creative text |
| Top-$k$ / top-$p$ | no | 1 forward | the default for chat and reasoning models, stacked with temperature |

- [Holtzman et al. 2020](https://arxiv.org/abs/1904.09751) showed that for open-ended
  text, maximizing likelihood (beam search) produces bland, repetitive output,
  while unrestricted sampling produces incoherent output; nucleus sampling sits between.
- For reasoning models, sampling is also what makes *multiple attempts* different
  from one another (lecture 21: majority voting, best-of-$n$).
- Decoding is a free knob: it changes behaviour without changing a single parameter.


# The modern decoder-only block

Tokens in, probabilities out: between them sits a stack of identical blocks. UDL
and lecture 10 describe the 2017 block. Every component of it has since been
swapped for something simpler or cheaper, and this section goes through the
swaps one at a time, because they are exactly the fields you will read in a
model's configuration file later in the lecture.


## The 2017 block and the 2026 block

![The 2017 encoder/decoder block of Vaswani et al. and the decoder-only block used by Llama, Qwen and Gemma. Generated with `instructor/fig-scripts/11-Transformer-Architecture-in-Practice/make_figures.py`.](figs/11-Transformer-Architecture-in-Practice/decoder-block-2017-vs-2026.png)

## What changed, and why

| Component | 2017 ([Vaswani et al.](https://arxiv.org/abs/1706.03762)) | 2026 (Llama 3, Qwen3, Gemma 3, DeepSeek-V3) | Reason |
|---|---|---|---|
| Norm placement | post-norm | **pre-norm** | stable training without warmup tricks at depth ([Xiong et al. 2020](https://arxiv.org/abs/2002.04745)) |
| Norm | LayerNorm | **RMSNorm** | same effect, no mean, no bias, cheaper ([Zhang and Sennrich 2019](https://arxiv.org/abs/1910.07467)) |
| Position | sinusoidal, added to input | **RoPE**, rotation of $\mathbf{q}$ and $\mathbf{k}$ | relative positions; extrapolates with scaling ([Su et al. 2021](https://arxiv.org/abs/2104.09864)) |
| Attention | multi-head, $n_h$ K/V heads | **GQA**, $n_{kv} \ll n_h$ | shrinks the KV cache ([Ainslie et al. 2023](https://arxiv.org/abs/2305.13245)) |
| MLP | ReLU, 2 matrices, width $4D$ | **SwiGLU**, 3 matrices, width $\approx 8D/3$ | better loss at equal parameters ([Shazeer 2020](https://arxiv.org/abs/2002.05202)) |
| Biases | everywhere | **none** | stability at scale ([Chowdhery et al. 2022](https://arxiv.org/abs/2204.02311)) |
| Dropout | 0.1 | **none** | one epoch over trillions of tokens does not overfit |
| MLP, large models | dense | **mixture of experts** | more parameters per FLOP ([Fedus et al. 2022 (JMLR; arXiv 2021)](https://arxiv.org/abs/2101.03961)) |

What did *not* change: residual connections, scaled dot-product attention, the
causal mask, and next-token cross-entropy.

## Pre-norm and RMSNorm

**Pre-norm**: normalize the *input* of each sublayer, not its output, so the
residual stream $\mathbf{h}$ is never normalized:

$$
\begin{aligned}
\mathbf{h} &\leftarrow \mathbf{h} + \mathrm{Attn}\big[\mathrm{RMSNorm}[\mathbf{h}]\big] \\
\mathbf{h} &\leftarrow \mathbf{h} + \mathrm{MLP}\big[\mathrm{RMSNorm}[\mathbf{h}]\big]
\end{aligned}
$$

The gradient has a clean path from the loss to every layer through the sum,
which is why pre-norm trains at depth where post-norm needs warmup and small
learning rates ([Xiong et al. 2020](https://arxiv.org/abs/2002.04745); lecture 08).


**RMSNorm** drops LayerNorm's mean subtraction and bias
([Zhang and Sennrich 2019](https://arxiv.org/abs/1910.07467)):

$$
\mathrm{RMSNorm}[\mathbf{h}] = \frac{\mathbf{h}}{\sqrt{\tfrac{1}{D}\sum_{d=1}^{D} h_d^2 + \epsilon}} \odot \mathbf{g}
$$

$D$ learned gains $\mathbf{g}$, one reduction instead of two. In the config:
`rms_norm_eps: 1e-6`. Gemma 2/3 add a second RMSNorm on each sublayer's output
as well ("sandwich" norm).


## RoPE in one slide

Rotary position embedding ([Su et al. 2021](https://arxiv.org/abs/2104.09864);
lecture 10) rotates each 2-D pair $(2i, 2i{+}1)$ of the query and key at position
$m$ by the angle $m\theta_i$:

$$
\theta_i = \Theta^{-2i/d_\text{head}}, \qquad i = 0, \dots, d_\text{head}/2 - 1
$$

- The dot product $\mathbf{q}_m^\top \mathbf{k}_n$ then depends on $n - m$ only (lecture 10's convention):
  **relative** position, with no learned parameters and nothing added to the embeddings.
- The pair $i = 0$ rotates fastest (one radian per token); the last pair has
  wavelength $2\pi\Theta$ tokens. The **base** $\Theta$ sets the longest wavelength:
  10,000 in the paper, 500,000 in Llama 3, 1,000,000 in Qwen3 and Gemma 3's global layers.
- A larger base is the first knob for longer context; `rope_scaling` (position
  interpolation, YaRN) is the second. Both appear in the config.


## SwiGLU: the gated MLP

The 2017 MLP is $\boldsymbol{\Omega}_2\,\mathrm{ReLU}[\boldsymbol{\Omega}_1 \mathbf{h} + \boldsymbol{\beta}_1] + \boldsymbol{\beta}_2$
with $\boldsymbol{\Omega}_1 \in \mathbb{R}^{4D \times D}$: $8D^2$ parameters.
[Shazeer 2020](https://arxiv.org/abs/2002.05202) replaces it with a *gated linear
unit* using the Swish (SiLU) activation $\mathrm{Swish}[z] = z\,\sigma[z]$:

$$
\mathrm{MLP}_\text{SwiGLU}[\mathbf{h}] = \boldsymbol{\Omega}_\text{down}
\Big( \mathrm{Swish}[\boldsymbol{\Omega}_\text{gate}\,\mathbf{h}] \odot \boldsymbol{\Omega}_\text{up}\,\mathbf{h} \Big),
\qquad \boldsymbol{\Omega}_\text{gate}, \boldsymbol{\Omega}_\text{up} \in \mathbb{R}^{d_\text{ff} \times D},\;
\boldsymbol{\Omega}_\text{down} \in \mathbb{R}^{D \times d_\text{ff}}
$$

- Three matrices, so $3\,D\,d_\text{ff}$ parameters. To match the 2017 budget of
  $8D^2$, set $d_\text{ff} = \tfrac{8}{3}D$: the **8/3 convention**. Llama 1 7B:
  $D = 4096$, $\tfrac{8}{3}D = 10{,}923$, rounded up to a multiple of 256 gives
  $d_\text{ff} = 11{,}008$ ([Touvron et al. 2023](https://arxiv.org/abs/2302.13971)).
- In practice models go a bit wider: Qwen3 uses $3D$, Llama 3 8B $3.5D$. The config
  fields are `intermediate_size` and `hidden_act: "silu"`.
- Why it helps is empirical: at equal parameters and compute, GLU variants reach
  lower perplexity in Shazeer's T5 ablations. The "divine benevolence" line in the
  paper is his explanation.
- FLOPs per token for the MLP: $2 \times 3\,D\,d_\text{ff}$, that is $2P$ for its $P$ parameters. (Symbols: this lecture writes $P$ for parameter counts and $N$ for sequence length; lecture 12's $6ND$ rule uses $N$ for parameters.)


The width convention as a calculation, and the SwiGLU MLP itself on a toy
vector. The 8/3 rule lands within 1% of the 2017 budget; the wider choices of
Qwen3 and Llama 3 spend 12% and 31% more. Change `D`, or try
`multiple_of = 64`.

In [ ]:
import math
import numpy as np

D = 4096            # hidden size (Llama 1 7B)
multiple_of = 256   # d_ff is rounded up to a multiple of this

def swiglu(h, W_gate, W_up, W_down):
    g = W_gate @ h
    return W_down @ (g / (1 + np.exp(-g)) * (W_up @ h))   # Swish[g] = g * sigmoid[g]

relu_mlp = 2 * D * 4 * D                                   # two matrices, width 4D
d_ff = multiple_of * math.ceil(8 * D / 3 / multiple_of)    # the 8/3 rule, rounded up
print(f"2017 ReLU MLP, d_ff = 4D = {4 * D:,}: {relu_mlp / 1e6:.1f}M parameters")
print(f"SwiGLU, 8D/3 = {8 * D / 3:,.1f} -> d_ff = {d_ff:,}: {3 * D * d_ff / 1e6:.1f}M "
      f"({3 * D * d_ff / relu_mlp:.3f}x)")
for name, ratio in [("Qwen3, d_ff = 3D", 3.0), ("Llama 3 8B, d_ff = 3.5D", 3.5)]:
    P = 3 * D * int(ratio * D)
    print(f"SwiGLU, {name} = {int(ratio * D):,}: {P / 1e6:.1f}M ({P / relu_mlp:.3f}x); "
          f"{2 * P / 1e6:.0f} MFLOP per token")

rng = np.random.default_rng(0)
D_toy, d_ff_toy = 6, 16                                    # 3 * 6 * 16 = 288 = 8 * 6**2
W_gate, W_up = rng.normal(size=(2, d_ff_toy, D_toy)) / np.sqrt(D_toy)
W_down = rng.normal(size=(D_toy, d_ff_toy)) / np.sqrt(d_ff_toy)
h = rng.normal(size=D_toy)
print("\ntoy SwiGLU:", np.round(h, 2), "->", np.round(swiglu(h, W_gate, W_up, W_down), 2))

## Small choices that add up

- **No biases.** PaLM removed every bias in dense layers and norms and reported
  better stability at scale ([Chowdhery et al. 2022](https://arxiv.org/abs/2204.02311));
  Llama, Gemma and Qwen3 follow (`attention_bias: false`). Qwen2 kept a bias on
  $\mathbf{q}, \mathbf{k}, \mathbf{v}$; Qwen3 dropped it and added **QK-norm**, an
  RMSNorm on $\mathbf{q}$ and $\mathbf{k}$ per head before the dot product, which
  bounds the attention logits ([Qwen Team 2025](https://arxiv.org/abs/2505.09388)).
- **Tied embeddings.** The input matrix $\boldsymbol{\Omega}_e$ ($D \times |\mathcal{V}|$)
  and the output projection ($|\mathcal{V}| \times D$) can share one matrix
  ([Press and Wolf 2017](https://arxiv.org/abs/1608.05859)). For Qwen3-0.6B that
  saves 156M parameters, 26% of the model; Qwen3-8B unties (`tie_word_embeddings: false`)
  because 0.6B of 8B is not worth the constraint. Gemma 3 ties at every size.
- **Attention output scaling and soft-capping** (Gemma 2) and **parallel
  attention + MLP** (PaLM, GPT-J) come and go; the items above are the stable core.
- **Weight precision.** `torch_dtype: "bfloat16"`: 2 bytes per parameter is the
  default for weights at rest; FP8 for DeepSeek-V3 (lecture 12).


# Attention variants

The attention sublayer is where the architecture choices with the largest
practical consequences live, because attention is the one part of the model
whose cost depends on the sequence length. This section starts from the KV
cache, derives its size, and then presents each variant as a way of shrinking it.


## Why decoding is different: the KV cache

Generating token $t$ needs $\mathbf{q}_t$ against the keys and values of
*every* earlier token. Those keys and values do not change (causal mask, no
dependence on later tokens), so we compute them once and keep them.

- **Prefill**: run the prompt through the model in one batched pass; store
  $\mathbf{K}, \mathbf{V}$ for every layer and head.
- **Decode**: one new token per step. Compute its $\mathbf{q}, \mathbf{k}, \mathbf{v}$,
  append $\mathbf{k}, \mathbf{v}$ to the cache, attend over the cache. Each step
  touches every weight once and the whole cache once, for a single token's worth of FLOPs.
- Decode is therefore **memory-bandwidth bound** (lecture 12's roofline): the
  time per token is roughly (weights + cache) / bandwidth. Shrinking the cache
  is the same as speeding up generation.
- Without the cache, step $t$ would recompute $t$ tokens' keys and values:
  $O(N^2)$ work per layer over a generation of length $N$.


## KV-cache memory per token: the derivation

Per layer, per key/value head, one token stores a key of $d_\text{head}$ numbers
and a value of $d_\text{head}$ numbers. With $L$ layers, $n_{kv}$ K/V heads and
$b$ bytes per number:

$$
\text{bytes per token} = \underbrace{2}_{\mathbf{k},\,\mathbf{v}} \times L \times n_{kv} \times d_\text{head} \times b
$$

**Llama 2 70B** ([Touvron et al. 2023](https://arxiv.org/abs/2307.09288)): $L = 80$,
$n_h = 64$ query heads, $n_{kv} = 8$, $d_\text{head} = 128$, bf16 ($b = 2$):

$$
2 \times 80 \times 8 \times 128 \times 2 = 327{,}680 \text{ bytes} = 320 \text{ KB per token}
$$

With one K/V head per query head (multi-head attention, $n_{kv} = 64$) it would
be **2.5 MB per token**. A 128K-token sequence: 40 GB versus 320 GB.


Serving a batch of $B$ sequences multiplies this by $B$. The cache, not the
weights, is what fills the GPU at long context.


## Multi-query and grouped-query attention

![Generated with `instructor/fig-scripts/11-Transformer-Architecture-in-Practice/make_figures.py`, after Figure 2 of Ainslie et al. 2023.](figs/11-Transformer-Architecture-in-Practice/attention-head-sharing.png)

- **MQA** ([Shazeer 2019](https://arxiv.org/abs/1911.02150)): all $n_h$ query heads
  share one $\mathbf{k}$ and one $\mathbf{v}$ projection. Cache shrinks by $n_h$; some quality loss.
- **GQA** ([Ainslie et al. 2023](https://arxiv.org/abs/2305.13245)): $n_{kv}$ K/V heads,
  each shared by $n_h / n_{kv}$ query heads. Cache shrinks by $n_h / n_{kv}$.
  With $n_{kv} = 8$ on T5-XXL, quality matched MHA at close to MQA speed.
- The attention computation is unchanged: each query head still does a full
  softmax over the sequence. Only the projections $\boldsymbol{\Omega}_k, \boldsymbol{\Omega}_v$
  are smaller ($n_{kv} d_\text{head} \times D$ instead of $n_h d_\text{head} \times D$).
- $n_{kv} = 8$ is also the number of GPUs in a node, so each tensor-parallel shard
  holds exactly one K/V head. Llama 3, Qwen3 and Mistral all use 8; Gemma 3 1B uses MQA.


## Sliding window and local-global alternation

**Sliding-window attention**: each token attends only to the previous $W$
tokens. The cache per layer is capped at $W$ entries (a rolling buffer), and
the receptive field still grows by $W$ per layer, to $L \times W$ at the top,
exactly like the dilated and banded patterns in UDL Figure 12.14.

- Mistral 7B: $W = 4096$ on every layer ([Jiang et al. 2023](https://arxiv.org/abs/2310.06825)).
- **Gemma 2**: alternate one local ($W = 4096$) and one global layer
  ([Gemma Team 2024](https://arxiv.org/abs/2408.00118)).
- **Gemma 3**: five local layers ($W = 1024$) for every global layer; only the
  global layers see the full 128K context
  ([Gemma Team 2025](https://arxiv.org/abs/2503.19786)).


Cache for a sequence of $N$ tokens with $L_g$ global and $L_\ell$ local layers:

$$
2\,n_{kv}\,d_\text{head}\,b \times \big(L_g N + L_\ell \min(N, W)\big)
$$

Gemma 3 27B ($L = 62$: 10 global, 52 local; $n_{kv} = 16$, $d_\text{head} = 128$)
at $N = 128$K:

- all-global: $62 \times 8$ KB $\times 131{,}072 = 62$ GB
- 5:1 local-global: $10 \times 8$ KB $\times 131{,}072 + 52 \times 8$ KB $\times 1024 = 10.4$ GB

Six times smaller, with the long-range path kept in every sixth layer.


## Multi-head latent attention (DeepSeek-V2/V3)

Instead of sharing heads, **compress** them. Cache one low-rank latent per token
and rebuild all heads' keys and values from it
([DeepSeek-AI 2024, DeepSeek-V2](https://arxiv.org/abs/2405.04434), section 2.1):

$$
\mathbf{c}_t = \boldsymbol{\Omega}_{DKV}\,\mathbf{h}_t \in \mathbb{R}^{d_c}, \qquad
\mathbf{k}_t^{(i)} = \boldsymbol{\Omega}_{UK}^{(i)}\,\mathbf{c}_t, \quad
\mathbf{v}_t^{(i)} = \boldsymbol{\Omega}_{UV}^{(i)}\,\mathbf{c}_t
$$

- The up-projections $\boldsymbol{\Omega}_{UK}, \boldsymbol{\Omega}_{UV}$ can be
  absorbed into the query and output projections, so the cache is never expanded
  at decode time.
- RoPE does not commute with that absorption, so a small **decoupled** rotary key
  of $d_h^R$ dims, shared by all heads, is cached alongside $\mathbf{c}_t$.


DeepSeek-V3 config: `kv_lora_rank: 512`, `qk_rope_head_dim: 64`, 128 heads of
$d_\text{head} = 128$, $L = 61$.

| | numbers cached per token per layer |
|---|---:|
| MHA (128 heads) | $2 \times 128 \times 128 = 32{,}768$ |
| GQA, 8 K/V heads | $2 \times 8 \times 128 = 2{,}048$ |
| MLA | $d_c + d_h^R = 512 + 64 = 576$ |

$576 \times 61 \times 2$ bytes $= 68.6$ KB per token for a 671B-parameter model,
less than Llama 3 8B's 128 KB. The paper calls this "equivalent to GQA with
2.25 groups", with quality above MHA in their ablations.


## KV cache versus context, pictured

![KV cache for one sequence in bf16, from each model's `config.json` and the bytes-per-token formula. Generated with `instructor/fig-scripts/11-Transformer-Architecture-in-Practice/make_figures.py`.](figs/11-Transformer-Architecture-in-Practice/kv-cache-vs-context.png)


# Mixture of experts

Attention variants shrink the cost of the sequence dimension. The mixture of
experts attacks the other axis: it lets a model have far more parameters than
any one token uses, which breaks the link between model size and compute per
token that lecture 12's $6ND$ rule (with $N$ = parameters there) assumes.


## Decouple parameters from compute per token

In a dense transformer every parameter touches every token: forward FLOPs per
token $\approx 2P$. More capacity means more compute, always.

- The MLP is about two thirds of a dense block's parameters, and it is applied
  to each token independently. Replace it with $E$ parallel MLPs (**experts**)
  and a small **router** that sends each token to $k$ of them.
- Compute per token scales with the $k$ experts that run; parameters scale
  with all $E$. The ratio $E / k$ is the free lunch, paid for in memory and communication.
- [Shazeer et al. 2017](https://arxiv.org/abs/1701.06538) did this for LSTMs
  (137B parameters in 2017); [GShard](https://arxiv.org/abs/2006.16668) and
  [Switch Transformer](https://arxiv.org/abs/2101.03961) (Fedus et al. 2022, JMLR; arXiv 2021) for
  transformers; Mixtral, DeepSeek-V3, Llama 4, Qwen3-235B and GPT-OSS are MoE.


## The router and top-$k$ gating

Following the notation of Switch Transformer ([Fedus et al. 2022 (JMLR; arXiv 2021)](https://arxiv.org/abs/2101.03961)),
adapted to column vectors. For a token's hidden vector $\mathbf{h} \in \mathbb{R}^D$:

$$
\mathbf{p} = \mathrm{softmax}[\boldsymbol{\Omega}_r\,\mathbf{h}] \in \mathbb{R}^E, \qquad
\mathcal{T} = \mathrm{TopK}[\mathbf{p},\, k], \qquad
\mathbf{y} = \sum_{i \in \mathcal{T}} p_i\,\mathrm{E}_i[\mathbf{h}]
$$

- $\boldsymbol{\Omega}_r \in \mathbb{R}^{E \times D}$ is the whole router: $E \cdot D$
  parameters, negligible.
- Each $\mathrm{E}_i$ is an ordinary SwiGLU MLP, usually narrower than a dense
  model's (DeepSeek-V3: $d_\text{ff} = 2048$ per expert against $18{,}432$ for its dense layers).
- The gate values $p_i$ multiply the expert outputs, so gradients reach the
  router through them; the TopK selection itself is not differentiable.
- $k = 1$ (Switch), $k = 2$ (Mixtral), $k = 8$ of 256 (DeepSeek-V3). DeepSeek adds
  $E_s$ **shared experts** that every token uses, so routed experts specialize
  ([Dai et al. 2024](https://arxiv.org/abs/2401.06066)).


## Routing, pictured

![One token through an 8-expert, top-2 MoE layer. Generated with `instructor/fig-scripts/11-Transformer-Architecture-in-Practice/make_figures.py`.](figs/11-Transformer-Architecture-in-Practice/moe-routing.png)

## Active versus total parameters

Split the parameter count into what every token touches and what is routed:

$$
P_\text{total} = P_\text{shared} + E \cdot P_\text{expert}, \qquad
P_\text{active} = P_\text{shared} + k \cdot P_\text{expert}
$$

where $P_\text{shared}$ is attention, norms, embeddings, routers and any shared experts.

**Mixtral 8x7B** ([Jiang et al. 2024](https://arxiv.org/abs/2401.04088)): $E = 8$,
$k = 2$, $D = 4096$, $d_\text{ff} = 14{,}336$, $L = 32$.

- One expert: $3 \times 4096 \times 14{,}336 = 176$M; per layer $8 \times 176$M $= 1.41$B of experts.
- Shared per layer: attention with GQA-8, $\approx 42$M. Embeddings in and out: $2 \times 32{,}000 \times 4096 = 262$M.
- $P_\text{total} \approx 32 \times (1.41\text{B} + 42\text{M}) + 262\text{M} = 46.7$B;
  $P_\text{active} \approx 32 \times (2 \times 176\text{M} + 42\text{M}) + 262\text{M} = 12.9$B.
- The paper's numbers: 46.7B total, 12.9B per token. Forward FLOPs per token
  $\approx 2 \times 12.9$B, a 13B model's cost with a 47B model's parameters.


## Load balancing

Left alone, the router collapses: a few experts get most tokens, improve, and
get more. Two fixes.

**Auxiliary loss** ([Fedus et al. 2022 (JMLR; arXiv 2021)](https://arxiv.org/abs/2101.03961), eq. 4):
with $f_i$ the fraction of tokens in the batch sent to expert $i$ and $P_i$ the
mean router probability for expert $i$,

$$
L_\text{aux} = \alpha\, E \sum_{i=1}^{E} f_i\, P_i, \qquad \alpha = 10^{-2}
$$

Minimized (value 1) when both are uniform at $1/E$. $f_i$ is not differentiable,
$P_i$ is, so the gradient pushes probability away from busy experts. Cost: the
loss competes with the language-modelling loss, and $\alpha$ must be tuned.


**Bias-based, no auxiliary loss** ([DeepSeek-AI 2024, V3](https://arxiv.org/abs/2412.19437),
section 2.1.2): compute affinities $s_i = \sigma[\mathbf{e}_i^\top \mathbf{h}]$
(sigmoid, not softmax) and select the top $k$ by $s_i + b_i$, but **weight the
output by $s_i$ alone**. After each step:

$$
b_i \leftarrow \begin{cases} b_i - \gamma & \text{if expert } i \text{ is overloaded} \\ b_i + \gamma & \text{if underloaded} \end{cases}
$$

with $\gamma = 0.001$. The bias steers *selection* only, so the model's
gradient is undisturbed; a tiny sequence-level balance loss ($\alpha = 10^{-4}$)
remains as a guard. In the config: `topk_method: "noaux_tc"`, `scoring_func: "sigmoid"`.


Both fixes on a toy router. The hidden vectors of a batch share a common
direction, as real ones do, so a random router already sends most tokens to two
experts, and the Switch balance term $E \sum_i f_i P_i$ (the auxiliary loss
without $\alpha$) is well above its minimum of 1. The second half applies
DeepSeek-V3's bias rule to the same batch, with the router weights frozen: only
the selection biases move. Nothing here is trained; the bias loop is the
per-step update repeated on one batch.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
n_tokens, D, E, k = 4096, 64, 8, 2       # a batch of hidden vectors, 8 experts, top-2
gamma, steps = 0.001, 1000               # DeepSeek-V3's bias step; number of updates

H = rng.normal(size=(n_tokens, D)) + rng.normal(size=D)   # tokens share a common direction
Omega_r = rng.normal(size=(E, D)) / np.sqrt(D)
logits = H @ Omega_r.T

def load(scores):
    chosen = np.argsort(scores, axis=1)[:, -k:]          # the top-k experts of each token
    return np.bincount(chosen.ravel(), minlength=E)

p = np.exp(logits - logits.max(1, keepdims=True))        # Switch: softmax router
p /= p.sum(1, keepdims=True)
counts = load(p)
f, P = counts / counts.sum(), p.mean(0)
print("tokens per expert, softmax top-2:      ", counts)
print(f"E * sum_i f_i P_i = {E * np.sum(f * P):.2f} (1.00 when balanced)")

s = 1 / (1 + np.exp(-logits))                            # DeepSeek-V3: sigmoid affinities
b = np.zeros(E)
for _ in range(steps):
    counts = load(s + b)                                 # select by s + b ...
    b -= gamma * np.sign(counts - counts.mean())         # ... and nudge b toward balance
print(f"tokens per expert after {steps} bias steps:", load(s + b))
print("biases b_i:", b.round(3))

## Worked example: DeepSeek-V3, 671B total, 37B active

From the [DeepSeek-V3 report](https://arxiv.org/abs/2412.19437) and its
`config.json`: $D = 7168$, $L = 61$ (first 3 dense, 58 MoE), 256 routed experts
plus 1 shared per MoE layer, $k = 8$, expert $d_\text{ff} = 2048$.

- **One expert** (SwiGLU): $3 \times 7168 \times 2048 = 44.0$M parameters.
- **Routed experts**: $58 \times 256 \times 44.0\text{M} = 654$B. **Shared experts**:
  $58 \times 44.0\text{M} = 2.6$B. So 656B of the 671B, 98%, are experts.
- **Everything every token touches**: MLA attention over 61 layers ($\approx 11.4$B),
  3 dense MLPs (1.2B), embeddings and output head (1.9B), routers (0.1B): $\approx 14.6$B.
- **Active per token**: $14.6\text{B} + 58 \times (8 + 1) \times 44.0\text{M} = 14.6 + 23.0 = 37.6$B.
  The paper says 37B. **Total**: $14.6 + 656.5 = 671$B. Both check.
- **Why it is cheaper per token**: forward FLOPs $\approx 2 \times 37$B $= 74$ GFLOP, against
  $1{,}342$ GFLOP for a dense 671B model, an 18$\times$ saving. Training on 14.8T tokens:
  $6 \times 37\text{B} \times 14.8\text{T} = 3.3 \times 10^{24}$ FLOPs. The report's 2.664M H800
  GPU-hours $\times$ 989 TFLOP/s (bf16 peak) $\approx 9.5 \times 10^{24}$ peak FLOPs, so this run
  achieved about 35% MFU against the bf16 peak (about 17% against FP8 peak); see lecture 12.
- **What is not cheaper**: memory. All 671B parameters must be resident: 671 GB
  in FP8, more than eight 80 GB GPUs for the weights alone.


Both worked examples from their configs. For DeepSeek-V3 the MLA attention
term is computed from its projections (`q_lora_rank: 1536`, `kv_lora_rank: 512`,
`qk_nope_head_dim: 128`, `qk_rope_head_dim: 64`, `v_head_dim: 128`) rather than
quoted; norms are left out, as on the slides. Change $k$ or $E$ and watch active
and total move independently.

In [ ]:
def swiglu_params(D, d_ff):
    return 3 * D * d_ff

# Mixtral 8x7B
D, L, E, k, d_ff, vocab = 4096, 32, 8, 2, 14336, 32000
n_h, n_kv, d_head = 32, 8, 128
expert = swiglu_params(D, d_ff)
attn = D * d_head * (2 * n_h + 2 * n_kv)                 # q and o; k and v (GQA)
shared = L * (attn + E * D) + 2 * vocab * D              # attention, routers, untied embeddings
total = shared + L * E * expert
active = shared + L * k * expert
print(f"Mixtral 8x7B: one expert {expert / 1e6:.0f}M, attention {attn / 1e6:.1f}M per layer")
print(f"  P_total {total / 1e9:.1f}B, P_active {active / 1e9:.1f}B ({total / active:.1f}x)")

# DeepSeek-V3
D, L, L_dense, E, E_shared, k = 7168, 61, 3, 256, 1, 8
d_ff_expert, d_ff_dense, vocab = 2048, 18432, 129280
n_h, q_rank, kv_rank, d_nope, d_rope, d_v = 128, 1536, 512, 128, 64, 128
mla = (D * q_rank + q_rank * n_h * (d_nope + d_rope)     # query down- and up-projection
       + D * (kv_rank + d_rope)                          # latent c_t and the shared RoPE key
       + kv_rank * n_h * (d_nope + d_v)                  # key and value up-projections
       + n_h * d_v * D)                                  # output projection
L_moe = L - L_dense
expert = swiglu_params(D, d_ff_expert)
shared = (L * mla + L_dense * swiglu_params(D, d_ff_dense)
          + 2 * vocab * D + L_moe * E * D)               # attention, dense MLPs, embeddings, routers
total = shared + L_moe * (E + E_shared) * expert
active = shared + L_moe * (k + E_shared) * expert
print(f"DeepSeek-V3: one expert {expert / 1e6:.1f}M, MLA {L * mla / 1e9:.1f}B, "
      f"touched by every token {shared / 1e9:.1f}B")
print(f"  P_total {total / 1e9:.0f}B, P_active {active / 1e9:.1f}B ({total / active:.0f}x); "
      f"experts are {(total - shared) / total:.0%} of the parameters")

## What MoE costs

- **Memory, not FLOPs, sets the hardware floor.** Total parameters must fit;
  a 37B-active model needs a 671B-sized cluster. Expert weights are spread over
  GPUs with **expert parallelism** (lecture 12).
- **All-to-all communication.** Each MoE layer dispatches every token to the
  GPUs holding its $k$ experts and collects the results: two all-to-all exchanges
  per layer. DeepSeek-V3 limits each token to at most 4 nodes and overlaps the
  exchange with compute to keep it hidden.
- **Batch size.** An expert is only compute-bound if enough tokens reach it per
  step (lecture 12's arithmetic intensity). With 256 experts and $k = 8$, a batch
  must hold thousands of tokens for every expert to get a few hundred. Small-batch
  inference of an MoE is bandwidth-bound and wasteful.
- **Training stability.** Routing is discrete; early MoEs diverged without
  router z-loss ([Zoph et al. 2022](https://arxiv.org/abs/2202.08906)) and careful
  precision. Fine-tuning can overfit more easily than a dense model.
- **Capacity factor and dropped tokens.** Implementations with fixed expert
  buffers drop tokens that overflow; DeepSeek-V3 drops none.


# Reading a real config

Every component so far corresponds to a field in the configuration file that
ships with an open-weight model. This section reads one such file top to
bottom, computes the model's parameter count from it, and checks the result
against what the model card says. You should be able to repeat this for any
model on Hugging Face.


## Qwen3-0.6B `config.json`

Fetched with `curl -sSfL https://huggingface.co/Qwen/Qwen3-0.6B/raw/main/config.json`
([Qwen Team 2025](https://arxiv.org/abs/2505.09388); Apache 2.0, runs on a laptop).

```json
{
  "architectures": ["Qwen3ForCausalLM"],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "max_position_embeddings": 40960,
  "max_window_layers": 28,
  "model_type": "qwen3",
```


```json
  "num_attention_heads": 16,
  "num_hidden_layers": 28,
  "num_key_value_heads": 8,
  "rms_norm_eps": 1e-06,
  "rope_scaling": null,
  "rope_theta": 1000000,
  "sliding_window": null,
  "tie_word_embeddings": true,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.51.0",
  "use_cache": true,
  "use_sliding_window": false,
  "vocab_size": 151936
}
```


## Reading the fields

| Field | Value | Symbol | What it tells you |
|---|---:|---|---|
| `architectures` | `Qwen3ForCausalLM` | | decoder-only, next-token objective |
| `num_hidden_layers` | 28 | $L$ | blocks in the stack |
| `hidden_size` | 1024 | $D$ | width of the residual stream |
| `num_attention_heads` | 16 | $n_h$ | query heads |
| `head_dim` | 128 | $d_\text{head}$ | $n_h d_\text{head} = 2048 \neq D$: the query projection *expands* |
| `num_key_value_heads` | 8 | $n_{kv}$ | GQA, 2 query heads per K/V head |
| `intermediate_size`, `hidden_act` | 3072, `silu` | $d_\text{ff}$ | SwiGLU MLP, $d_\text{ff} = 3D$ |
| `rope_theta`, `rope_scaling` | 1,000,000, none | $\Theta$ | RoPE base; YaRN would appear here for 128K |
| `max_position_embeddings` | 40,960 | $N_\text{max}$ | context the model was trained to |
| `sliding_window` | none | $W$ | every layer is global |
| `vocab_size`, `tie_word_embeddings` | 151,936, true | $|\mathcal{V}|$ | one embedding matrix, used in and out |
| `attention_bias`, `rms_norm_eps` | false, $10^{-6}$ | | no biases; RMSNorm $\epsilon$ |
| `torch_dtype` | bfloat16 | $b$ | 2 bytes per parameter on disk |

## Parameter count from the config

- **Embedding** (tied, counted once): $|\mathcal{V}| \times D = 151{,}936 \times 1024 = 155.6$M
- **Attention, per layer** (no biases):
  $\boldsymbol{\Omega}_q$: $D \times n_h d_\text{head} = 1024 \times 2048 = 2.10$M;
  $\boldsymbol{\Omega}_k, \boldsymbol{\Omega}_v$: $D \times n_{kv} d_\text{head} = 1024 \times 1024 = 1.05$M each;
  $\boldsymbol{\Omega}_o$: $n_h d_\text{head} \times D = 2.10$M. Total **6.29M**.
- **MLP, per layer**: $3 \times D \times d_\text{ff} = 3 \times 1024 \times 3072 =$ **9.44M**.
- **Norms, per layer**: two RMSNorms ($2 \times 1024$) and QK-norm ($2 \times 128$): 2,304. Negligible.
- **Per layer**: $6.29 + 9.44 + 0.002 = 15.73$M. Attention is 40% of a block, the MLP 60%.
- **All layers**: $28 \times 15.73\text{M} = 440.5$M. Final RMSNorm: 1,024.
- **Total**: $155.6\text{M} + 440.5\text{M} =$ **596.0M**.


**Check**: the Qwen3 release lists Qwen3-0.6B as 0.6B parameters, 0.44B
non-embedding ([Qwen Team 2025](https://arxiv.org/abs/2505.09388), Table 1).
Both match. Untied, it would be 752M.


## What the count tells you

- **Where the parameters are**: 26% embedding, 30% attention, 44% MLP. In Qwen3-8B
  ($D = 4096$, $L = 36$, untied) the split is 15% embeddings, 85% blocks: the
  embedding share falls as $D$ and $L$ grow, which is why small models tie.
- **Weights in memory**: $596\text{M} \times 2$ bytes $= 1.1$ GB in bf16.
- **KV cache per token**: $2 \times 28 \times 8 \times 128 \times 2 = 112$ KB. At the
  full 40,960-token context: **4.4 GB**, four times the weights. Without GQA
  ($n_{kv} = 16$) it would be 8.8 GB.
- **FLOPs per token**: $2 \times 440\text{M}$ for the blocks plus $2 \times 156\text{M}$
  for the output projection $\approx 1.2$ GFLOP, plus attention over the context.
  One H100 could in principle serve about $10^6$ tokens per second of this model;
  in practice it serves far fewer, because decode is bandwidth-bound.
- The same arithmetic on Qwen3-8B gives 8.19B total (6.95B non-embedding) and
  144 KB of cache per token. Try it; the config is at the same URL with `8B`.



The whole count from the config, as a function of the dictionary (the JSON
above, as Python). Memory is in binary units, so the slides' 112 KB and 4.4 GB
are KiB and GiB. The last lines rerun it on Qwen3-8B by overriding five fields.
Paste in another model's config, or set `"num_key_value_heads": 1` for MQA.

In [ ]:
config = {                                   # Qwen3-0.6B config.json
    "architectures": ["Qwen3ForCausalLM"], "attention_bias": False, "attention_dropout": 0.0,
    "bos_token_id": 151643, "eos_token_id": 151645, "head_dim": 128, "hidden_act": "silu",
    "hidden_size": 1024, "initializer_range": 0.02, "intermediate_size": 3072,
    "max_position_embeddings": 40960, "max_window_layers": 28, "model_type": "qwen3",
    "num_attention_heads": 16, "num_hidden_layers": 28, "num_key_value_heads": 8,
    "rms_norm_eps": 1e-06, "rope_scaling": None, "rope_theta": 1000000, "sliding_window": None,
    "tie_word_embeddings": True, "torch_dtype": "bfloat16", "transformers_version": "4.51.0",
    "use_cache": True, "use_sliding_window": False, "vocab_size": 151936,
}
bytes_per_value = {"bfloat16": 2, "float16": 2, "float32": 4}
KiB, GiB = 2**10, 2**30

def count(cfg):
    D, L, V = cfg["hidden_size"], cfg["num_hidden_layers"], cfg["vocab_size"]
    n_h, n_kv, d_head = cfg["num_attention_heads"], cfg["num_key_value_heads"], cfg["head_dim"]
    attn = D * d_head * (2 * n_h + 2 * n_kv)              # q and o; k and v; no biases
    mlp = 3 * D * cfg["intermediate_size"]                # SwiGLU: gate, up, down
    norms = 2 * D + 2 * d_head                            # two RMSNorms and QK-norm
    blocks = L * (attn + mlp + norms) + D                 # plus the final RMSNorm
    embed = V * D * (1 if cfg["tie_word_embeddings"] else 2)
    return dict(attn=attn, mlp=mlp, embed=embed, blocks=blocks, total=embed + blocks,
                attn_all=L * attn, mlp_all=L * mlp)

def kv_per_token(cfg):                                    # bytes: 2 x L x n_kv x d_head x b
    return (2 * cfg["num_hidden_layers"] * cfg["num_key_value_heads"] * cfg["head_dim"]
            * bytes_per_value[cfg["torch_dtype"]])

c = count(config)
print(f"per layer: attention {c['attn'] / 1e6:.2f}M, MLP {c['mlp'] / 1e6:.2f}M")
print(f"embedding {c['embed'] / 1e6:.1f}M + non-embedding {c['blocks'] / 1e6:.1f}M "
      f"= {c['total'] / 1e6:.1f}M; untied: "
      f"{count({**config, 'tie_word_embeddings': False})['total'] / 1e6:.0f}M")
print(f"shares: embedding {c['embed'] / c['total']:.0%}, attention {c['attn_all'] / c['total']:.0%}, "
      f"MLP {c['mlp_all'] / c['total']:.0%}")

weights = c["total"] * bytes_per_value[config["torch_dtype"]]
N_max = config["max_position_embeddings"]
mha = {**config, "num_key_value_heads": config["num_attention_heads"]}
print(f"weights {weights / GiB:.2f} GiB; KV cache {kv_per_token(config) / KiB:.0f} KiB per token, "
      f"{kv_per_token(config) * N_max / GiB:.2f} GiB at {N_max:,} tokens "
      f"({kv_per_token(config) * N_max / weights:.1f}x the weights); "
      f"without GQA {kv_per_token(mha) * N_max / GiB:.2f} GiB")

qwen3_8b = {**config, "hidden_size": 4096, "num_hidden_layers": 36, "num_attention_heads": 32,
            "intermediate_size": 12288, "tie_word_embeddings": False}
c8 = count(qwen3_8b)
print(f"\nQwen3-8B: {c8['total'] / 1e9:.2f}B total, {c8['blocks'] / 1e9:.2f}B non-embedding "
      f"(embeddings {c8['embed'] / c8['total']:.0%}), KV cache {kv_per_token(qwen3_8b) / KiB:.0f} KiB per token")

# Context length

The knobs met in this lecture all show up again when a model's context is
extended, which is now a separate stage of training with its own costs. One
slide to connect them.


## Why context costs what it costs, and the knobs

**Three costs of a longer context $N$**

1. **KV cache**: linear in $N$ per sequence, slope $2 L n_{kv} d_\text{head} b$.
   At 128K this exceeds the weights for most models.
2. **Attention FLOPs**: the $\mathbf{Q}\mathbf{K}^\top$ and $\mathbf{A}\mathbf{V}$
   products cost $4 L N n_h d_\text{head}$ per token, so *prefill* of a prompt is
   quadratic in $N$ and each decode step is linear in $N$. FlashAttention (lecture 12)
   removes the $N \times N$ memory, not the FLOPs.
3. **Position generalization**: RoPE at positions beyond those seen in training
   gives attention patterns the model never learned. Extending context is a
   *training* stage (Llama 3: six stages from 8K to 128K).


**The knobs, and which cost each one moves**

| Knob | Moves |
|---|---|
| GQA, MLA | cache slope (1) |
| sliding window, local-global | cache slope and FLOPs on most layers (1, 2) |
| larger RoPE base $\Theta$ | position generalization (3) |
| RoPE scaling: position interpolation ([Chen et al. 2023](https://arxiv.org/abs/2306.15595)), YaRN ([Peng et al. 2023](https://arxiv.org/abs/2309.00071)) | (3), with little or no extra training |
| hybrids: linear-attention or SSM layers with occasional full attention (lecture 09) | (1) and (2): constant state per layer |
| cache quantization, paging (lecture 15) | the $b$ factor and fragmentation |

DeepSeek-V3's config shows the scaling route: `rope_scaling: {"type": "yarn",
"factor": 40, "original_max_position_embeddings": 4096}`, taking 4K to 160K.


# Summary

One line per idea; the recap deck uses these.


## Summary

- Tokenizers map text to IDs from a fixed vocabulary built by byte-pair encoding: start from the 256 bytes, repeatedly merge the most frequent adjacent pair; every string is representable, and many "model" failures are tokenizer failures.
- UTF-8 encodes 1,114,112 Unicode code points in one to four bytes; byte-level BPE works on those bytes, so the base vocabulary is 256.
- The embedding matrix is $D \times |\mathcal{V}|$ and can be a quarter of a small model; tying it with the output projection halves that.
- Decoding turns a probability vector into a token: greedy and beam search maximize likelihood (translation), temperature plus top-$k$/top-$p$ sampling trades accuracy for diversity (chat, reasoning); decoding is a knob that changes behaviour without changing parameters.
- The 2026 block is pre-norm with RMSNorm, RoPE on $\mathbf{q}$ and $\mathbf{k}$, grouped-query attention, a SwiGLU MLP of width about $8D/3$ (three matrices, $3 D d_\text{ff}$ parameters), no biases and no dropout.
- KV cache per token is $2 \times L \times n_{kv} \times d_\text{head} \times$ bytes; decode is bandwidth-bound, so shrinking the cache is the same as faster generation.
- MQA and GQA share K/V heads across query heads (cache down by $n_h / n_{kv}$); sliding windows cap the cache of local layers at $W$; MLA caches a $d_c + d_h^R$ latent per token instead of all heads.
- A mixture of experts replaces the MLP with $E$ experts and a router that picks $k$: $P_\text{active} = P_\text{shared} + k P_\text{expert}$ sets compute, $P_\text{total} = P_\text{shared} + E P_\text{expert}$ sets memory; DeepSeek-V3 is 671B total, 37B active, 18$\times$ cheaper per token than a dense 671B.
- Load balancing keeps the router from collapsing: an auxiliary loss $\alpha E \sum_i f_i P_i$ (Switch) or a per-expert selection bias updated by $\pm\gamma$ (DeepSeek-V3).
- A `config.json` gives $L$, $D$, $n_h$, $d_\text{head}$, $n_{kv}$, $d_\text{ff}$, $|\mathcal{V}|$ and tying; from them the parameter count is $|\mathcal{V}| D\,(\times 2 \text{ if untied}) + L\,[D\,d_\text{head}(2 n_h + 2 n_{kv}) + 3 D d_\text{ff}]$, which gives 596M for Qwen3-0.6B.
- Context costs a linear KV cache, quadratic prefill FLOPs, and position generalization; GQA/MLA and windows move the first two, RoPE base and scaling the third, hybrids all three.

## Further reading

- [Vaswani et al., *Attention Is All You Need* (2017)](https://arxiv.org/abs/1706.03762) next to [Grattafiori et al., *The Llama 3 Herd of Models* (2024)](https://arxiv.org/abs/2407.21783), section 3.2: the 2017 and 2024 blocks described by their authors.
- [Karpathy, *Let's build the GPT Tokenizer* (2024)](https://github.com/karpathy/minbpe/blob/master/lecture.md): BPE from scratch in Python, two hours well spent.
- [Ainslie et al., *GQA* (2023)](https://arxiv.org/abs/2305.13245) and [DeepSeek-AI, *DeepSeek-V2* (2024)](https://arxiv.org/abs/2405.04434), section 2.1: the two ways to shrink the KV cache.
- [Fedus, Zoph and Shazeer, *Switch Transformers* (2022, JMLR; arXiv 2021)](https://arxiv.org/abs/2101.03961) and [DeepSeek-AI, *DeepSeek-V3 Technical Report* (2024)](https://arxiv.org/abs/2412.19437), section 2: MoE routing and load balancing, five years apart.
- [Gemma Team, *Gemma 3 Technical Report* (2025)](https://arxiv.org/abs/2503.19786), section 2: local-global attention and its KV-cache accounting.
- [Hugging Face `transformers`, `modeling_qwen3.py`](https://github.com/huggingface/transformers/blob/main/src/transformers/models/qwen3/modeling_qwen3.py): every field of the config, in about 500 lines of PyTorch.